# Module 4.3 - Selective Memory

**Reference:** [`03-selective-memory.md`](./03-selective-memory.md)

## What you'll do in this notebook

- Score each message's importance with a fast heuristic and with an LLM.
- Drop the lowest-importance messages first when you need to free up tokens.
- Verify that the account-number test survives aggressive pruning with the right scoring.

## Setup

In [ ]:
import os
import re
from dotenv import load_dotenv
from openai import OpenAI
import tiktoken

load_dotenv()
assert os.getenv("OPENAI_API_KEY"), "OPENAI_API_KEY is not set."

client = OpenAI()
MODEL = os.getenv("OPENAI_MODEL", "gpt-4o-mini")
encoding = tiktoken.encoding_for_model("gpt-4o-mini")
print(f"OK: client ready, model = {MODEL}")

## Exercise 1 - Heuristic importance scoring

Rule-based scoring is fast and free. Start at a neutral 5 and nudge up or down based on keywords.

In [ ]:
HIGH = [
    r"\b(account|id|number|email|phone)\b",
    r"\b(ticket|order|reference)\s*#?\d+",
    r"\b(deadline|urgent|critical|asap)\b",
    r"\b(error|bug|issue|problem)\b",
    r"^\d+[.:]",
]
LOW = [
    r"^(thanks?|thx|ty|thank you)\b",
    r"^(ok|okay|sure|yes|no)[.! ]?$",
    r"^(hi|hello|hey)[,! ]?$",
    r"\b(lol|haha|hmm)\b",
]

def heuristic_score(message: str) -> int:
    score = 5
    # TODO:
    # 1. For each pattern in HIGH, if re.search(pattern, message, re.IGNORECASE) matches, add 2.
    # 2. For each pattern in LOW, subtract 2 if it matches.
    # 3. If len(message.split()) > 30, add 1 (longer messages tend to carry more substance).
    # 4. Clamp to [1, 10] and return.
    raise NotImplementedError

for m in [
    "Thanks!",
    "My account ID is ACC-42.",
    "Sure.",
    "I'm getting error code 500 on the login page.",
    "Hello",
    "I need this resolved by Friday.",
    "Just chatting about the weather for a bit, no real question",
]:
    print(f"{heuristic_score(m):2}  {m}")

## Exercise 2 - LLM-based importance scoring

Slower and pricier, but handles nuance the regexes can't. Useful as a fallback or as a high-quality labelling pass.

In [ ]:
SCORE_SYS = (
    "You rate how important a chat message is for maintaining conversation context. "
    "1-3 = pleasantries or filler. 4-6 = relevant but not critical. "
    "7-10 = facts, decisions, numbers, or identifiers that must be retained. "
    "Reply with the integer only."
)

def llm_score(message: str) -> int:
    # TODO: call the chat API with SCORE_SYS as system and the message as user,
    # temperature=0, max_tokens=4. Parse the reply as an int, clamp to [1, 10],
    # and return 5 on any parse failure.
    raise NotImplementedError

for m in ["Thanks for checking", "My order number is 918273", "I'll think about it."]:
    print(f"heuristic={heuristic_score(m):2}   llm={llm_score(m):2}   {m!r}")

## Exercise 3 - Selective pruning

Instead of dropping the *oldest* messages when the token budget is exceeded, drop the *least important* ones. Always keep the most recent pair - the active exchange.

In [ ]:
class SelectiveMemoryChatbot:
    def __init__(self, system_prompt: str, max_tokens: int = 400):
        self.system_prompt = system_prompt
        self.max_tokens = max_tokens
        self.history: list[dict] = []  # each dict has role, content, importance

    def _prune(self) -> None:
        def tokens() -> int:
            return sum(len(encoding.encode(m["content"])) for m in self.history)

        # TODO:
        # While tokens() > self.max_tokens and len(self.history) > 2:
        #   Consider every message EXCEPT the last two (the active pair).
        #   Find the one with the lowest importance.
        #   Remove it AND its partner (user<->assistant pair).
        raise NotImplementedError

    def chat(self, user_message: str) -> str:
        self.history.append({
            "role": "user",
            "content": user_message,
            "importance": heuristic_score(user_message),
        })

        messages = [{"role": "system", "content": self.system_prompt}]
        messages += [{"role": m["role"], "content": m["content"]} for m in self.history]
        reply = client.chat.completions.create(model=MODEL, messages=messages).choices[0].message.content

        self.history.append({
            "role": "assistant",
            "content": reply,
            "importance": heuristic_score(reply),
        })
        self._prune()
        return reply

bot = SelectiveMemoryChatbot("You are a helpful support assistant. Keep replies short.", max_tokens=250)
bot.chat("My account ID is ACC-42.")
bot.chat("Thanks!")
bot.chat("OK, go ahead.")
for i in range(6):
    bot.chat(f"Filler message number {i}, just chatting.")

print("History kept after pruning (role / importance / content):")
for m in bot.history:
    print(f"  {m['role']:<9} imp={m['importance']}  {m['content'][:80]}")

print("\nRecall test:", bot.chat("What is my account ID?"))

The 'My account ID is ACC-42' message should have survived thanks to its high importance score - the pruning targeted the 'Thanks!' and 'OK' messages first.

## What's next

Scoring keeps important turns but still only looks inside the current session. `04-retrieval-based-memory.ipynb` lets the bot recall turns from any past session by storing them in a vector database.